# LAB01 · SVM con pipeline, búsqueda de parámetros y evaluación

**Pregunta de investigación.** ¿Puede un clasificador SVM con núcleo RBF, después de estandarizar
las variables dentro de un pipeline, distinguir tumores malignos de benignos a partir de las
medidas de núcleos-citopatología del Wisconsin, y qué margen de error es aceptable si el error más
costoso es diagnosticar como benigno un tumor maligno?

**Unidad de análisis.** Un **tumor** (una observación del dataset es una muestra citopatológica
etiquetada por un patólogo; no es una persona ni un episodio clínico).

**Variable objetivo (target).** `target`: 0 = maligno, 1 = benigno.

**Advertencia.** Este es un conjunto de demostración de propósito didáctico, con datos históricos de
Wisconsin y una sola institución. Un buen desempeño **no** demuestra capacidad diagnóstica en un
entorno clínico real, no valida un sistema de apoyo al diagnóstico y no debe usarse para decidir
tratamientos. Sirve para practicar la construcción correcta de un pipeline y la lectura crítica de
sus resultados.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u01.data import audit_dataframe, audit_table, load_breast_cancer
from inf8239_u01.evaluation import (
    confusion_matrix_frame,
    cv_table,
    plot_confusion,
    plot_cv_results,
    report_frame,
    stratified_split,
)
from inf8239_u01.models import build_baseline, build_svm

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
print("Proyecto:", ROOT)

Proyecto: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01


## 1. Auditoría previa del dataset (antes de modelar)

Se revisa el tipo de cada variable, los valores ausentes, los duplicados y el balance de clases.
Entrenar sin esta revisión puede producir métricas engañosas.

In [2]:
features, target = load_breast_cancer()
data = features.copy()
data["target"] = target.to_numpy()

print("Forma de X:", features.shape)
print("Clases del target:", target.value_counts().to_dict())
print("Proporción de clases:", (target.value_counts(normalize=True) * 100).round(2).to_dict(), "%")

audit = audit_dataframe(data, "target")
print("\nNulos totales:", audit["total_missing"], "| Filas duplicadas:", audit["duplicate_rows"])
print("Columnas con más de un valor único:", sum(1 for value in audit["unique_per_column"].values() if value > 1))

tabla_auditoria = audit_table(data)
tabla_auditoria.head(8)

Forma de X: (569, 30)
Clases del target: {1: 357, 0: 212}
Proporción de clases: {1: 62.74, 0: 37.26} %

Nulos totales: 0 | Filas duplicadas: 0
Columnas con más de un valor único: 31


,columna,tipo,nulos,porcentaje_nulos,valores_unicos,ejemplo
0,mean radius,float64,0,0.0,456,17.99
1,mean texture,float64,0,0.0,479,10.38
2,mean perimeter,float64,0,0.0,522,122.8
3,mean area,float64,0,0.0,539,1001.0
4,mean smoothness,float64,0,0.0,474,0.1184
5,mean compactness,float64,0,0.0,537,0.2776
6,mean concavity,float64,0,0.0,537,0.3001
7,mean concave points,float64,0,0.0,542,0.1471


In [3]:
# El target nunca debe formar parte de las variables de entrada
assert "target" not in features.columns
print("¿El target está dentro de X?", "target" in features.columns)
print("Tipos de las variables:", features.dtypes.value_counts().to_dict())

¿El target está dentro de X? False
Tipos de las variables: {dtype('float64'): 30}


## 2. Partición estratificada 80/20 con semilla 42

La partición se hace **antes** de cualquier transformación. La estandarización y la imputación se
aprenden dentro del pipeline, sobre el conjunto de entrenamiento; así no se filtra información del
conjunto de prueba.

In [4]:
X_train, X_test, y_train, y_test = stratified_split(features, target)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Balance en entrenamiento:", y_train.value_counts().to_dict())
print("Balance en prueba:", y_test.value_counts().to_dict())
print("Proporción de clase malignant en prueba: %.4f" % (y_test.value_counts(normalize=True).get(0)))

Entrenamiento: (455, 30) | Prueba: (114, 30)
Balance en entrenamiento: {1: 285, 0: 170}
Balance en prueba: {1: 72, 0: 42}
Proporción de clase malignant en prueba: 0.3684


## 3. Baseline y SVM con pipeline

- **Baseline**: `DummyClassifier(strategy="most_frequent")` predice siempre la clase mayoritaria
  (benigno). Es el mínimo que un modelo útil debe superar.
- **SVM**: `Pipeline(StandardScaler + SVC(rbf))`. El escalado vive dentro del pipeline.

In [5]:
baseline = build_baseline()
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

modelo = build_svm(C=1.0, gamma="scale")
modelo.fit(X_train, y_train)
svm_pred = modelo.predict(X_test)

comparativo = pd.DataFrame(
    [
        {
            "modelo": "DummyClassifier (baseline)",
            "accuracy": accuracy_score(y_test, baseline_pred),
            "f1_macro": f1_score(y_test, baseline_pred, average="macro"),
            "recall_maligno": recall_score(y_test, baseline_pred, pos_label=0),
        },
        {
            "modelo": "SVC rbf (C=1, gamma=scale)",
            "accuracy": accuracy_score(y_test, svm_pred),
            "f1_macro": f1_score(y_test, svm_pred, average="macro"),
            "recall_maligno": recall_score(y_test, svm_pred, pos_label=0),
        },
    ]
)
comparativo

,modelo,accuracy,f1_macro,recall_maligno
0,DummyClassifier (baseline),0.631579,0.387097,0.00000
1,"SVC rbf (C=1, gamma=scale)",0.982456,0.981151,0.97619


## 4. Búsqueda de parámetros con validación cruzada

`GridSearchCV` con `StratifiedKFold(5, shuffle=True, random_state=42)` y `scoring="f1_macro"`.
F1 macro porque las clases no están perfectamente balanceadas y el desempeño promedio no debe
depender de la clase mayoritaria.

In [6]:
param_grid = {"model__C": [0.1, 1.0, 10.0], "model__gamma": ["scale", 0.01, 0.1]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busqueda = GridSearchCV(
    estimator=build_svm(),
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    refit=True,
)
busqueda.fit(X_train, y_train)

tabla_cv = cv_table(busqueda.cv_results_)
tabla_cv.to_csv(REPORTS / "svm_cv_results.csv", index=False)
tabla_cv

,C,gamma,f1_macro_mean,f1_macro_std,fit_time_mean_s,fit_time_std_s,rank_test_score
0,10.0,0.01,0.973888,0.017677,0.010833,0.002801,1
1,1.0,0.01,0.969041,0.016283,0.006882,0.000695,2
2,10.0,scale,0.967086,0.013638,0.007969,0.002376,3
3,1.0,scale,0.966940,0.015966,0.020338,0.005885,4
4,1.0,0.1,0.953323,0.012853,0.008156,0.000662,5
5,0.1,0.01,0.942078,0.014347,0.027035,0.004774,6
6,0.1,scale,0.937715,0.018035,0.033365,0.016575,7
7,10.0,0.1,0.937532,0.023718,0.010920,0.004030,8
8,0.1,0.1,0.934135,0.015687,0.026113,0.007363,9


In [7]:
plot_cv_results(tabla_cv, REPORTS / "svm_cv_results.png",
                "F1 macro en validación por configuración (5 folds, semilla 42)")
print("Guardado:", REPORTS / "svm_cv_results.png")

Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/svm_cv_results.png


## 5. Evaluación del mejor modelo sobre el conjunto de prueba

El modelo elegido por la búsqueda (`best_estimator_`) se evalúa una sola vez sobre la prueba, que
no intervino en ninguna búsqueda.

In [8]:
mejor = busqueda.best_estimator_
predicciones = mejor.predict(X_test)

print("Mejores parámetros:", busqueda.best_params_)
print("Mejor F1 macro en validación: %.5f" % busqueda.best_score_)

reporte = report_frame(y_test, predicciones, target_names=["maligno", "benigno"])
reporte

Mejores parámetros: {'model__C': 10.0, 'model__gamma': 0.01}
Mejor F1 macro en validación: 0.97389


,clase,precision,recall,f1_score,support
0,maligno,0.976190,0.976190,0.976190,42.0
1,benigno,0.986111,0.986111,0.986111,72.0
2,macro avg,0.981151,0.981151,0.981151,114.0
3,weighted avg,0.982456,0.982456,0.982456,114.0


In [9]:
matriz = confusion_matrix_frame(y_test, predicciones, labels=[0, 1], target_names=["maligno", "benigno"])
print(matriz)

plot_confusion(y_test, predicciones, labels=[0, 1], target_names=["maligno", "benigno"],
               path=REPORTS / "svm_confusion_matrix.png",
               title="SVM (mejor configuración) · matriz de confusión")
print("Guardado:", REPORTS / "svm_confusion_matrix.png")

              pred_maligno  pred_benigno
real_maligno            41             1
real_benigno             1            71
Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/svm_confusion_matrix.png


## 6. Interpretación de los errores

En este problema la clase 0 es **maligno**:

- **Falso negativo** (real maligno, predicho benigno): el tumor se clasifica como benigno → se
  retrasa el tratamiento. Es el error más costoso.
- **Falso positivo** (real benigno, predicho maligno): se indique biopsia o seguimiento innecesario →
  ansiedad y coste sanitario, pero reversible.

Por eso el recall de la clase maligno es la métrica que más importa, aunque no la única: un recall
alto con muchos falsos positivos no es útil en un triaje.

In [10]:
# Matriz: filas = real, columnas = predicho, orden [maligno(0), benigno(1)]
M = confusion_matrix_frame(
    y_test, predicciones, labels=[0, 1], target_names=["maligno", "benigno"]
).to_numpy()

tp_maligno = int(M[0, 0])   # maligno predicho maligno
fn_maligno = int(M[0, 1])   # maligno predicho benigno  -> falso negativo costoso
fp_maligno = int(M[1, 0])   # benigno predicho maligno
tn_maligno = int(M[1, 1])   # benigno predicho benigno

metricas = {
    "baseline_accuracy": float(accuracy_score(y_test, baseline_pred)),
    "baseline_f1_macro": float(f1_score(y_test, baseline_pred, average="macro")),
    "baseline_recall_maligno": float(recall_score(y_test, baseline_pred, pos_label=0)),
    "accuracy": float(accuracy_score(y_test, predicciones)),
    "f1_macro": float(f1_score(y_test, predicciones, average="macro")),
    "recall_maligno": float(tp_maligno / (tp_maligno + fn_maligno)) if (tp_maligno + fn_maligno) else 0.0,
    "recall_benigno": float(tn_maligno / (tn_maligno + fp_maligno)) if (tn_maligno + fp_maligno) else 0.0,
    "especificidad": float(tn_maligno / (tn_maligno + fp_maligno)) if (tn_maligno + fp_maligno) else 0.0,
    "verdaderos_positivos_maligno": tp_maligno,
    "falsos_negativos_maligno": fn_maligno,
    "falsos_positivos_benigno": fp_maligno,
    "verdaderos_negativos_benigno": tn_maligno,
    "mejores_parametros": {key: str(value) for key, value in busqueda.best_params_.items()},
    "f1_macro_validacion": float(busqueda.best_score_),
    "seed": 42,
    "filas_entrenamiento": int(len(X_train)),
    "filas_prueba": int(len(X_test)),
}
(REPORTS / "svm_metrics.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
print(json.dumps(metricas, indent=2))

{
  "baseline_accuracy": 0.631578947368421,
  "baseline_f1_macro": 0.3870967741935484,
  "baseline_recall_maligno": 0.0,
  "accuracy": 0.9824561403508771,
  "f1_macro": 0.9811507936507937,
  "recall_maligno": 0.9761904761904762,
  "recall_benigno": 0.9861111111111112,
  "especificidad": 0.9861111111111112,
  "verdaderos_positivos_maligno": 41,
  "falsos_negativos_maligno": 1,
  "falsos_positivos_benigno": 1,
  "verdaderos_negativos_benigno": 71,
  "mejores_parametros": {
    "model__C": "10.0",
    "model__gamma": "0.01"
  },
  "f1_macro_validacion": 0.9738878509919948,
  "seed": 42,
  "filas_entrenamiento": 455,
  "filas_prueba": 114
}


In [11]:
joblib.dump(mejor, REPORTS / "svm_best.joblib")
print("Modelo guardado en:", REPORTS / "svm_best.joblib")

# Verificación funcional: el modelo guardado vuelve a predecir
modelo_cargado = joblib.load(REPORTS / "svm_best.joblib")
print("Predicciones idénticas tras recargar:", bool((modelo_cargado.predict(X_test) == predicciones).all()))

Modelo guardado en: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/svm_best.joblib
Predicciones idénticas tras recargar: True


## 7. Conclusión

El SVM con kernel RBF y estandarización dentro del pipeline supera de forma amplia al baseline
`DummyClassifier`: el baseline solo alcanza 0,6316 de exactitud y 0,3871 de F1 macro (predice siempre
la clase mayoritaria, benigno), mientras que el SVM alcanza 0,9825 de exactitud y 0,9812 de F1 macro
en el conjunto de prueba, con 41 de 42 casos malignos correctamente detectados (un falso negativo).
La búsqueda de hiperparámetros sobre 5 folds estratificados (C=10, gamma=0.01) mejora el F1 macro de
validación (0,9739) y el modelo elegido se evalúa una sola vez sobre el conjunto de prueba, que no
participó en la búsqueda.

El punto crítico no es el número sino su lectura: en este problema el falso negativo (tumour
maligno clasificado como benigno) es el error más costoso, y el recall de la clase maligno es la
métrica que mejor refleja ese riesgo. El desempeño reportado corresponde a un benchmark histórico
de propósito didáctico, con una sola institución y sin validación externa; **no** demuestra
capacidad diagnóstica y no debe generalizarse a decisiones clínicas. La decisión responsable es
usar estos resultados para evaluar el pipeline y el procedimiento, y no para sustituir el criterio
profesional.